In [6]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict,Annotated, Literal
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
import operator
import os

from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.4
) 

In [2]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [3]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [4]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'cake'}, config=config1)

{'topic': 'cake',
 'joke': 'Why did the cake go to therapy? \n\nBecause it was feeling crumby.',
 'explanation': 'The joke "Why did the cake go to therapy? Because it was feeling crumby" is a play on words. It uses a pun to create humor.\n\nIn this joke, "crumby" has a double meaning:\n\n1. In a literal sense, a cake can become crumby when it gets old or stale, and its texture changes into small, crumbling pieces.\n2. In an idiomatic sense, "feeling crumby" is a common expression used to describe someone who is feeling unwell or down, often with a stomachache or nausea.\n\nThe joke relies on this wordplay to create a pun, where the cake, which is a food item, is said to be "feeling crumby" in the idiomatic sense, implying that it\'s emotionally distressed, rather than just physically stale. The punchline is humorous because it takes a common expression and applies it to an inanimate object, a cake, which is not typically associated with emotions or feelings.'}

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'cake', 'joke': 'Why did the cake go to therapy? \n\nBecause it was feeling crumby.', 'explanation': 'The joke "Why did the cake go to therapy? Because it was feeling crumby" is a play on words. It uses a pun to create humor.\n\nIn this joke, "crumby" has a double meaning:\n\n1. In a literal sense, a cake can become crumby when it gets old or stale, and its texture changes into small, crumbling pieces.\n2. In an idiomatic sense, "feeling crumby" is a common expression used to describe someone who is feeling unwell or down, often with a stomachache or nausea.\n\nThe joke relies on this wordplay to create a pun, where the cake, which is a food item, is said to be "feeling crumby" in the idiomatic sense, implying that it\'s emotionally distressed, rather than just physically stale. The punchline is humorous because it takes a common expression and applies it to an inanimate object, a cake, which is not typically associated with emotions or feelings.'}, next=

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'cake', 'joke': 'Why did the cake go to therapy? \n\nBecause it was feeling crumby.', 'explanation': 'The joke "Why did the cake go to therapy? Because it was feeling crumby" is a play on words. It uses a pun to create humor.\n\nIn this joke, "crumby" has a double meaning:\n\n1. In a literal sense, a cake can become crumby when it gets old or stale, and its texture changes into small, crumbling pieces.\n2. In an idiomatic sense, "feeling crumby" is a common expression used to describe someone who is feeling unwell or down, often with a stomachache or nausea.\n\nThe joke relies on this wordplay to create a pun, where the cake, which is a food item, is said to be "feeling crumby" in the idiomatic sense, implying that it\'s emotionally distressed, rather than just physically stale. The punchline is humorous because it takes a common expression and applies it to an inanimate object, a cake, which is not typically associated with emotions or feelings.'}, next

In [8]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to therapy?\n\nBecause it was feeling a little "twisted."',
 'explanation': 'This joke is a play on words, which is a common technique used in humor. The punchline "feeling a little \'twisted\'" has a double meaning here.\n\nOn one hand, "twisted" can refer to the physical shape of the spaghetti, which is a long, curved, and twisted piece of pasta. This is a literal interpretation of the word.\n\nOn the other hand, "twisted" can also mean emotionally or mentally disturbed, which is a common usage in the context of therapy. So, when the joke says the spaghetti is "feeling a little \'twisted\'," it\'s making a pun on the word, implying that the spaghetti is not just physically twisted but also emotionally troubled, hence the need for therapy.\n\nThe joke relies on this wordplay to create a humorous connection between the setup (spaghetti going to therapy) and the punchline (feeling a little \'twisted\'). It\'s a lighthearted and cleve

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'trump', 'joke': 'Why did Donald Trump bring a ladder to the party?\n\nBecause he heard the drinks were on the house.', 'explanation': 'This joke relies on a play on words and a common phrase. The phrase "the drinks are on the house" typically means that the drinks are being provided for free by the establishment or host. In this joke, Donald Trump brings a ladder to the party, which is an unexpected and humorous action.\n\nThe punchline "because he heard the drinks were on the house" is a double entendre. On one hand, it\'s a literal interpretation of the phrase, implying that the drinks are literally on the house, and Trump needs a ladder to reach them. On the other hand, it\'s a clever play on words, referencing the common phrase while also making a humorous connection to the ladder.\n\nThe joke is likely making fun of Donald Trump\'s perceived ego and self-importance, suggesting that he thinks he\'s so tall or important that he needs a ladder to reach

## Fault tolerence

In [7]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [3]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [4]:
def step_1(state: CrashState) -> CrashState:
    print("Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("Step 3 executed")
    return {"done": True}

In [8]:
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.add_edge(START,'step_1')
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:

try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
Step 1 executed
Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
